# Score Validation Framework

This notebook validates that the quality scoring methodology produces sensible, defensible results before scores are shown to anyone.

**Approach:**
1. **Face validity** — Do high-scoring providers look right on manual review? Do safety gate failures get caught?
2. **Sensitivity analysis** — How much do composite rankings change when we adjust dimension weights?
3. **Known-quality benchmarks** — Do providers with known disciplinary actions, board certifications, or MIPS scores land where we expect?

**Usage:** Data loading is commented out because source files are on Google Drive (too large for git). Populate the file paths in Cell 2, uncomment the `pd.read_*` lines, and run all cells.

In [ ]:
"""
Score Validation Framework
==========================
Face validity, sensitivity analysis, edge case review.
"""
import pandas as pd
import numpy as np

# Data loading placeholders - paths will be filled when data files are available
# step1_output = pd.read_parquet('step1_safety_gate_output.parquet')
# step2_output = pd.read_parquet('step2_output_final.parquet')
# wv_discipline = pd.read_excel('wv_discipline.xlsx')
# sample_npis = pd.read_csv('sample_npi_list.csv')

print("Validation framework loaded. Populate data paths when files are available.")

## Face Validity Checks

Face validity asks: "Does this pass the smell test?"

We cross-reference scoring outputs against known ground truth:
- Providers who **failed** the safety gate (Step 1) should not have high composite scores.
- Providers with **disciplinary actions** (WV board data) should be flagged or score low.
- Providers with **high MIPS scores** (Tier 1 data) should not be safety gate failures.
- Providers with **thin data** should be flagged as low-confidence, not scored as low-quality.

In [ ]:
# Face Validity: Cross-reference safety gate failures with MIPS scores
# Question: Does any NPI that FAILED the safety gate have a high MIPS score?

def check_safety_gate_mips_consistency(step1_df, step2_df):
    """Check if safety gate failures have suspiciously high MIPS scores."""
    failed_npis = step1_df[step1_df['safety_gate'] == 'fail']['npi']
    failed_with_mips = step2_df[step2_df['npi'].isin(failed_npis)]
    high_mips_failures = failed_with_mips[failed_with_mips['mips_final_score'] > 80]
    
    result = {
        'total_safety_failures': len(failed_npis),
        'failures_with_mips': len(failed_with_mips),
        'failures_with_high_mips': len(high_mips_failures),
        'concern': len(high_mips_failures) > 0
    }
    
    if result['concern']:
        print(f"\u26a0\ufe0f  CONCERN: {result['failures_with_high_mips']} providers failed safety gate but have MIPS > 80")
    else:
        print(f"\u2705 No safety gate failures have high MIPS scores")
    
    return result

# Uncomment when data is available:
# consistency = check_safety_gate_mips_consistency(step1_output, step2_output)

## Sensitivity Analysis: Composite Weights

The composite score is a weighted average of five dimensions. Since we have no outcome data to calibrate weights, we test whether the **rank order** of providers is stable across reasonable weight variations.

If small weight changes cause large rank swings, the composite is fragile and we need to either:
- Narrow the weight range we consider reasonable, or
- Disclose that rankings are approximate, not precise.

**Scenarios tested:**
- `baseline`: Equal weights (20% each)
- `mips_downweighted`: MIPS at 5% (per brainstorm critique about high clustering)
- `util_heavy`: Utilization at 35% (if we trust billing data most)
- `cred_heavy`: Credentials at 35% (if we trust board certification most)

In [ ]:
# Sensitivity Analysis: How much does composite rank order change with different weights?

weight_scenarios = {
    'baseline': {'mips': 0.20, 'util': 0.20, 'cred': 0.20, 'patient': 0.20, 'access': 0.20},
    'mips_downweighted': {'mips': 0.05, 'util': 0.25, 'cred': 0.25, 'patient': 0.25, 'access': 0.20},
    'util_heavy': {'mips': 0.10, 'util': 0.35, 'cred': 0.20, 'patient': 0.15, 'access': 0.20},
    'cred_heavy': {'mips': 0.10, 'util': 0.20, 'cred': 0.35, 'patient': 0.15, 'access': 0.20},
}

def compute_composite(scores_df, weights):
    """Compute composite score for a set of weights."""
    composite = pd.Series(0.0, index=scores_df.index)
    available_weight = 0.0
    
    for dim, weight in weights.items():
        col = f'{dim}_score'
        if col in scores_df.columns:
            mask = scores_df[col].notna()
            composite[mask] += scores_df.loc[mask, col] * weight
            available_weight += weight
    
    # Redistribute weight from missing dimensions
    if available_weight > 0:
        composite = composite / available_weight * sum(weights.values())
    
    return composite

def rank_stability_analysis(scores_df, scenarios):
    """Test how stable provider rankings are across weight scenarios."""
    rankings = {}
    for name, weights in scenarios.items():
        composite = compute_composite(scores_df, weights)
        rankings[name] = composite.rank(ascending=False)
    
    ranking_df = pd.DataFrame(rankings)
    
    # Compute max rank change per provider
    ranking_df['max_rank_change'] = ranking_df.max(axis=1) - ranking_df.min(axis=1)
    
    print(f"Rank stability across {len(scenarios)} scenarios:")
    print(f"  Median max rank change: {ranking_df['max_rank_change'].median():.0f}")
    print(f"  95th percentile rank change: {ranking_df['max_rank_change'].quantile(0.95):.0f}")
    print(f"  Providers with >20 rank change: {(ranking_df['max_rank_change'] > 20).sum()}")
    
    return ranking_df

# Uncomment when data is available:
# rank_results = rank_stability_analysis(all_scores_df, weight_scenarios)

## Next Steps

Validation activities to complete when data is available:

1. **Populate data paths** in Cell 2 and uncomment the `pd.read_*` lines.
2. **Run face validity checks** — review the output of `check_safety_gate_mips_consistency()` and investigate any flagged providers.
3. **Run sensitivity analysis** — evaluate rank stability and decide on final weight configuration.
4. **Populate edge case catalog** — identify at least 10 real providers matching the edge case templates in `edge_case_catalog.md`.
5. **Test known-disciplined providers** — cross-reference WV board discipline data against composite scores.
6. **Document threshold sensitivity** — for each arbitrary threshold in Step 3, test \u00b110% and \u00b120% shifts.
7. **Update `validation_results.md`** — check off each item as validation is completed.
8. **Review caveats** — finalize the list of caveats that must be disclosed before any external launch.